## Импорты инсталлы

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os
import warnings

pd.set_option('display.max_columns', 100)
warnings.simplefilter(action='ignore', category=FutureWarning)

In [23]:
load_dotenv()

False

In [24]:
API_KEY = "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"
print(f"API Key: {API_KEY is not None}")

API Key: True


## Описание

| Название колонки              | Описание                                                             |
| ----------------------------- | -------------------------------------------------------------------- |
| `id`                          | Уникальный идентификатор объявления.                                 |
| `name`                        | Название объявления.                                                 |
| `description`                 | Текстовое описание объявления.                                       |
| `property_type`               | Тип недвижимости, например apartment, house, private room.           |
| `room_type`                   | Тип сдаваемого помещения, например Entire home/apt или Private room. |
| `accommodates`                | Сколько гостей может разместить объект.                              |
| `bathrooms`                   | Информация о ванных комнатах.                                        |
| `bedrooms`                    | Информация о спальнях.                                               |
| `beds`                        | Количество кроватей.                                                 |
| `amenities`                   | Список удобств, предоставляемых в объекте.                           |
| `host_id`                     | Уникальный идентификатор хозяина.                                    |
| `host_name`                   | Имя хозяина.                                                         |
| `host_since`                  | Дата регистрации хозяина на Airbnb.                                  |
| `host_location`               | Местоположение хозяина.                                              |
| `host_about`                  | Информация хозяина о себе.                                           |
| `host_response_time`          | Время ответа хозяина.                                                |
| `host_response_rate`          | Доля ответов хозяина.                                                |
| `host_acceptance_rate`        | Доля подтвержденных хозяином бронирований.                           |
| `host_is_superhost`           | Является ли хозяин супер-хозяином.                                   |
| `host_neighbourhood`          | Район проживания хозяина, если указан.                               |
| `host_listings_count`         | Количество объявлений у хозяина.                                     |
| `host_total_listings_count`   | Общее число объявлений хозяина.                                      |
| `host_verifications`          | Способы верификации хозяина.                                         |
| `host_has_profile_pic`        | Есть ли у хозяина фото профиля.                                      |
| `host_identity_verified`      | Подтверждена ли личность хозяина.                                    |
| `latitude`                    | Географическая широта объекта.                                       |
| `longitude`                   | Географическая долгота объекта.                                      |
| `neighbourhood_overview`      | Описание района, где находится объект.                               |
| `city`                        | Город, где расположен объект.                                        |
| `price`                       | Цена за ночь.                                                        |
| `has_availability`            | Есть ли у объекта доступные даты.                                    |
| `availability_30`             | Количество доступных дней в ближайшие 30 дней.                       |
| `availability_60`             | Количество доступных дней в ближайшие 60 дней.                       |
| `availability_90`             | Количество доступных дней в ближайшие 90 дней.                       |
| `availability_365`            | Количество доступных дней в ближайшие 365 дней.                      |
| `availability_eoy`            | Доступность объекта на конец года.                                   |
| `number_of_reviews`           | Общее количество отзывов.                                            |
| `number_of_reviews_ltm`       | Количество отзывов за последние 12 месяцев.                          |
| `number_of_reviews_l30d`      | Количество отзывов за последние 30 дней.                             |
| `number_of_reviews_ly`        | Количество отзывов за последний год.                                 |
| `first_review`                | Дата первого отзыва.                                                 |
| `last_review`                 | Дата последнего отзыва.                                              |
| `review_scores_rating`        | Общий рейтинг по отзывам.                                            |
| `review_scores_accuracy`      | Оценка точности описания объекта.                                    |
| `review_scores_cleanliness`   | Оценка чистоты.                                                      |
| `review_scores_checkin`       | Оценка удобства заселения.                                           |
| `review_scores_communication` | Оценка качества коммуникации с хозяином.                             |
| `review_scores_location`      | Оценка расположения.                                                 |
| `review_scores_value`         | Оценка соотношения цены и качества.                                  |
| `reviews_per_month`           | Среднее количество отзывов в месяц.                                  |
| `estimated_occupancy_l365d`   | Оценочная заполняемость за последние 365 дней.                       |
| `estimated_revenue_l365d`     | Оценочная выручка за последние 365 дней.                             |


## Обзор

In [25]:
data = pd.read_csv("rent_predictions/airbnb_train.csv")
data.head()

FileNotFoundError: [Errno 2] No such file or directory: 'rent_predictions/airbnb_train.csv'

In [26]:
data.info()

NameError: name 'data' is not defined

In [27]:
data.duplicated().sum()

NameError: name 'data' is not defined

In [28]:
data.isna().sum().sort_values(ascending=False).loc[lambda x: x > 0]

NameError: name 'data' is not defined

Как мы видим в датасете есть пропуски во многих колонках, нет дубликатов, многие признаки, например `amenities` или любое `description` требует предварительной обработки, чтобы их можно было использовать 

Также можно заметить, что некоторые признаки нельзя использовать т.к они являются признаками полученнным постфактум

## Modelling agent

In [29]:
"""
LangGraph-based Modeling Agent.

Architecture:
- Modeling Supervisor selects primary metric and creates 3+ model workers.
- Model workers run in parallel.
- Each worker asks LLM to generate Python code for exactly one model.
- Python executor runs generated code.
- Judge selects the best model.
- Optional tuning branch improves the best model.
- Packaging node saves final modeling summary.

Expected input dataset:
- CSV file with prepared features.
- Target column must be present in the CSV.

Main public function:
    run_modeling_agent(...)

Example:
    result = run_modeling_agent(
        llm=llm,
        prepared_dataset_path="artifacts/prepared_dataset.csv",
        target_column="target",
        task_type="classification",
        business_task="Predict customer churn",
        run_id="demo_run_001"
    )
"""

from __future__ import annotations

import ast
import json
import os
import re
import shutil
import subprocess
import sys
import textwrap
import uuid
from datetime import datetime
from operator import add
from pathlib import Path
from typing import Any, Dict, List, Optional, TypedDict

from typing_extensions import Annotated

try:
    from langgraph.graph import StateGraph, START, END
except ImportError as exc:
    raise ImportError(
        "LangGraph is not installed. Install it with: pip install langgraph"
    ) from exc

try:
    # Newer LangGraph versions commonly expose Send here.
    from langgraph.types import Send
except ImportError:
    # Fallback for some older versions.
    from langgraph.constants import Send


# ============================================================
# 1. State schemas
# ============================================================

class ModelWorkerSpec(TypedDict, total=False):
    worker_id: str
    model_name: str
    task_type: str
    primary_metric: str
    rationale: str


class ModelWorkerResult(TypedDict, total=False):
    worker_id: str
    model_name: str
    status: str
    metrics: Dict[str, Any]
    model_path: Optional[str]
    predictions_path: Optional[str]
    metrics_path: Optional[str]
    code_path: Optional[str]
    error: Optional[str]
    generated_code: Optional[str]


class ModelingState(TypedDict, total=False):
    # Input
    run_id: str
    prepared_dataset_path: str
    target_column: str
    task_type: str
    business_task: str
    artifacts_dir: str

    # Optional upstream summaries
    data_summary: Dict[str, Any]
    prep_summary: Dict[str, Any]

    # Supervisor decisions
    primary_metric: str
    metric_reason: str
    worker_specs: List[ModelWorkerSpec]

    # Used only inside parallel worker execution
    current_worker_spec: ModelWorkerSpec

    # Parallel outputs.
    # Reducer is important: parallel workers should append results instead of overwriting.
    worker_results: Annotated[List[ModelWorkerResult], add]
    logs: Annotated[List[Dict[str, Any]], add]

    # Judge result
    best_model_name: str
    best_worker_id: str
    best_model_path: str
    best_predictions_path: str
    best_metrics_path: str
    best_metrics: Dict[str, Any]

    # Tuning
    tuning_used: bool
    tuning_status: str
    retry_count: int
    max_retries: int

    # Final artifacts
    metrics_path: str
    predictions_path: str
    modeling_summary_path: str
    final_model_path: str

    # Status
    status: str
    reason: Optional[str]


# ============================================================
# 2. Allowed models
# ============================================================

ALLOWED_MODELS = {
    "classification": [
        "LogisticRegression",
        "RandomForestClassifier",
        "GradientBoostingClassifier",
        "HistGradientBoostingClassifier",
        "KNeighborsClassifier",
        "DecisionTreeClassifier",
    ],
    "regression": [
        "LinearRegression",
        "Ridge",
        "RandomForestRegressor",
        "GradientBoostingRegressor",
        "HistGradientBoostingRegressor",
        "KNeighborsRegressor",
        "DecisionTreeRegressor",
    ],
}


# ============================================================
# 3. Utility functions
# ============================================================

def now_iso() -> str:
    return datetime.utcnow().isoformat(timespec="seconds") + "Z"


def make_log(agent: str, message: str, level: str = "INFO") -> Dict[str, Any]:
    return {
        "ts": now_iso(),
        "agent": agent,
        "level": level,
        "message": message,
    }


def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def safe_json_load(path: str | Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def safe_json_dump(payload: Dict[str, Any], path: str | Path) -> None:
    path = Path(path)
    ensure_dir(path.parent)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=4, ensure_ascii=False)


def extract_text_from_llm_response(response: Any) -> str:
    """
    Works with many LangChain chat model responses.
    """
    if response is None:
        return ""

    if isinstance(response, str):
        return response

    if hasattr(response, "content"):
        return str(response.content)

    return str(response)


def extract_json_from_text(text: str) -> Dict[str, Any]:
    """
    Extracts JSON from raw LLM output.
    Supports:
    - pure JSON
    - ```json ... ```
    - text with embedded {...}
    """
    text = text.strip()

    fenced = re.search(r"```(?:json)?\s*(.*?)```", text, flags=re.DOTALL)
    if fenced:
        text = fenced.group(1).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Try to find first JSON object.
    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        candidate = text[start:end + 1]
        return json.loads(candidate)

    raise ValueError(f"Could not parse JSON from LLM output:\n{text[:1000]}")


def extract_python_code(text: str) -> str:
    """
    Extracts Python code from LLM output.
    Supports:
    - pure Python
    - ```python ... ```
    """
    text = text.strip()

    fenced = re.search(r"```(?:python)?\s*(.*?)```", text, flags=re.DOTALL)
    if fenced:
        return fenced.group(1).strip()

    return text


def validate_generated_code_safety(code: str, output_dir: str) -> None:
    """
    Basic static safety validation.

    This is not a perfect sandbox. It is a practical guardrail for a student project.
    For production, run code in a real sandbox/container with stricter permissions.
    """

    forbidden_imports = {
        "requests",
        "urllib",
        "httpx",
        "socket",
        "ftplib",
        "paramiko",
        "shutil",  # avoid deleting/copying arbitrary paths from generated code
        "pathlib",  # optional: can be allowed, but here we keep it strict
    }

    forbidden_calls = {
        "eval",
        "exec",
        "compile",
        "__import__"
    }

    forbidden_strings = [
        "os.remove",
        "os.rmdir",
        "os.system",
        "subprocess",
        "rm -rf",
        "../",
        "~",
    ]

    for bad in forbidden_strings:
        if bad in code:
            raise ValueError(f"Generated code contains forbidden pattern: {bad}")

    try:
        tree = ast.parse(code)
    except SyntaxError as exc:
        raise ValueError(f"Generated code has syntax error: {exc}") from exc

    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                root_name = alias.name.split(".")[0]
                if root_name in forbidden_imports:
                    raise ValueError(f"Forbidden import in generated code: {alias.name}")

        if isinstance(node, ast.ImportFrom):
            module = node.module or ""
            root_name = module.split(".")[0]
            if root_name in forbidden_imports:
                raise ValueError(f"Forbidden import in generated code: {module}")

        if isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name):
                if node.func.id in forbidden_calls:
                    raise ValueError(f"Forbidden function call: {node.func.id}")

            if isinstance(node.func, ast.Attribute):
                full_attr = node.func.attr
                if full_attr in forbidden_calls:
                    raise ValueError(f"Forbidden function call: {full_attr}")


def run_python_file(
    code_path: str | Path,
    timeout_seconds: int = 180,
) -> Dict[str, Any]:
    """
    Executes a generated Python script in a subprocess.
    """
    code_path = str(code_path)

    proc = subprocess.run(
        [sys.executable, code_path],
        capture_output=True,
        text=True,
        timeout=timeout_seconds,
    )

    return {
        "returncode": proc.returncode,
        "stdout": proc.stdout,
        "stderr": proc.stderr,
        "status": "success" if proc.returncode == 0 else "error",
    }


def call_llm(llm: Any, prompt: str) -> str:
    """
    Minimal adapter for LangChain-compatible chat models.
    """
    response = llm.invoke(prompt)
    return extract_text_from_llm_response(response)


def get_worker_artifact_dir(state: ModelingState, worker_id: str) -> Path:
    return ensure_dir(Path(state["artifacts_dir"]) / state["run_id"] / "workers" / worker_id)


def get_run_artifact_dir(state: ModelingState) -> Path:
    return ensure_dir(Path(state["artifacts_dir"]) / state["run_id"])


def normalize_metric_name(metric: str) -> str:
    return metric.strip().lower().replace("-", "_").replace(" ", "_")


# ============================================================
# 4. Prompts
# ============================================================

def build_supervisor_prompt(state: ModelingState) -> str:
    task_type = state["task_type"]
    allowed = ALLOWED_MODELS.get(task_type, [])

    return f"""
You are a Modeling Supervisor inside a LangGraph multi-agent ML system.

Your job:
1. Analyze the ML task.
2. Choose the primary metric.
3. Create at least 3 independent model workers.
4. Each worker must train exactly one model.
5. Choose models only from the allowed model list.
6. Prefer diversity:
   - one simple baseline,
   - one tree-based model,
   - one boosting or alternative model.
7. Return only valid JSON.

Task type:
{task_type}

Business task:
{state.get("business_task", "")}

Target column:
{state["target_column"]}

Data summary:
{json.dumps(state.get("data_summary", {}), indent=2, ensure_ascii=False)}

Preprocessing summary:
{json.dumps(state.get("prep_summary", {}), indent=2, ensure_ascii=False)}

Allowed models:
{json.dumps(allowed, ensure_ascii=False)}

Metric guidance:
- For classification:
  - If class imbalance is likely, prefer f1_weighted.
  - Otherwise accuracy is acceptable.
  - You may also choose roc_auc only for binary classification if appropriate.
- For regression:
  - Prefer rmse when large errors are important.
  - Prefer mae when interpretability of average error is important.
  - r2 can be reported but should usually not be the only primary metric.

Return JSON with exactly this schema:
{{
  "primary_metric": "metric_name",
  "metric_reason": "why this metric is appropriate",
  "worker_specs": [
    {{
      "worker_id": "worker_short_unique_id",
      "model_name": "AllowedModelName",
      "task_type": "{task_type}",
      "primary_metric": "metric_name",
      "rationale": "why this model is useful"
    }}
  ]
}}

Important:
- Create at least 3 workers.
- Do not select models outside the allowed list.
- Return JSON only.
""".strip()


def build_worker_code_prompt(
    state: ModelingState,
    worker_spec: ModelWorkerSpec,
    worker_dir: Path,
) -> str:
    model_name = worker_spec["model_name"]
    task_type = state["task_type"]
    primary_metric = state["primary_metric"]

    return f"""
You are a Model Worker in a LangGraph ML system.

Generate a complete Python script to train exactly one assigned model.

Assigned model:
{model_name}

Task type:
{task_type}

Primary metric:
{primary_metric}

Dataset CSV path:
{state["prepared_dataset_path"]}

Target column:
{state["target_column"]}

Output directory:
{str(worker_dir)}

Strict requirements:
1. Use only these libraries:
   - pandas
   - numpy
   - json
   - os
   - joblib
   - sklearn
2. Do not use internet.
3. Do not read or write outside the output directory except reading the dataset CSV.
4. Train exactly this model: {model_name}
5. Load the dataset from the CSV path.
6. Split the data into train, validation and test:
   - train: 70%
   - validation: 15%
   - test: 15%
7. For classification, use stratify when possible.
8. Fit the model on train.
9. Evaluate on validation and test.
10. Save:
   - model: model.joblib
   - metrics: metrics.json
   - predictions: predictions.csv
11. metrics.json must include:
   {{
     "model_name": "...",
     "task_type": "...",
     "primary_metric": "...",
     "validation_metrics": {{}},
     "test_metrics": {{}}
   }}
12. predictions.csv must include:
   - y_true
   - y_pred
13. Print a compact JSON summary to stdout.
14. Return only Python code. No markdown.

Metric requirements:
- Classification metrics:
  accuracy, precision_weighted, recall_weighted, f1_weighted.
  If binary classification and predict_proba exists, also compute roc_auc.
- Regression metrics:
  mae, rmse, r2.

Implementation hints:
- Import the exact sklearn class needed for {model_name}.
- Use random_state=42 where supported.
- Use max_iter=1000 for LogisticRegression.
- Use reasonable defaults; do not run hyperparameter tuning in this worker.
""".strip()


def build_fix_code_prompt(
    state: ModelingState,
    worker_spec: ModelWorkerSpec,
    old_code: str,
    stderr: str,
    worker_dir: Path,
) -> str:
    return f"""
You are a code repair agent.

The previous generated Python code failed.

Your task:
- Fix the code.
- Keep the same assigned model.
- Preserve all output requirements.
- Return only corrected Python code.

Assigned worker:
{json.dumps(worker_spec, indent=2, ensure_ascii=False)}

Dataset CSV path:
{state["prepared_dataset_path"]}

Target column:
{state["target_column"]}

Output directory:
{str(worker_dir)}

Error stderr:
{stderr[-4000:]}

Previous code:
{old_code}

Return only Python code. No markdown.
""".strip()


def build_tuning_code_prompt(state: ModelingState, tuning_dir: Path) -> str:
    return f"""
You are a hyperparameter tuning worker in a LangGraph ML system.

Generate a complete Python script to tune the current best model.

Best model:
{state["best_model_name"]}

Task type:
{state["task_type"]}

Primary metric:
{state["primary_metric"]}

Dataset CSV path:
{state["prepared_dataset_path"]}

Target column:
{state["target_column"]}

Output directory:
{str(tuning_dir)}

Strict requirements:
1. Use only pandas, numpy, json, os, joblib, sklearn.
2. Do not use internet.
3. Do not write outside output directory.
4. Tune only the best model type: {state["best_model_name"]}.
5. Use GridSearchCV or RandomizedSearchCV.
6. Keep search space small, maximum about 12 parameter combinations.
7. Split train/test with test_size=0.2.
8. Save:
   - tuned model: model.joblib
   - metrics: metrics.json
   - predictions: predictions.csv
9. metrics.json must include:
   {{
     "model_name": "...",
     "task_type": "...",
     "primary_metric": "...",
     "best_params": {{}},
     "test_metrics": {{}}
   }}
10. predictions.csv must include y_true and y_pred.
11. Print compact JSON summary to stdout.
12. Return only Python code. No markdown.

Metric requirements:
- Classification:
  accuracy, precision_weighted, recall_weighted, f1_weighted.
  If binary classification and predict_proba exists, also compute roc_auc.
- Regression:
  mae, rmse, r2.
""".strip()


# ============================================================
# 5. Nodes
# ============================================================

def modeling_supervisor_node_factory(llm: Any):
    def modeling_supervisor_node(state: ModelingState) -> Dict[str, Any]:
        prompt = build_supervisor_prompt(state)
        raw = call_llm(llm, prompt)
        decision = extract_json_from_text(raw)

        primary_metric = normalize_metric_name(decision["primary_metric"])
        worker_specs = decision["worker_specs"]

        allowed = set(ALLOWED_MODELS[state["task_type"]])

        cleaned_specs: List[ModelWorkerSpec] = []
        seen_worker_ids = set()
        seen_models = set()

        for idx, spec in enumerate(worker_specs):
            model_name = spec.get("model_name")

            if model_name not in allowed:
                continue

            worker_id = spec.get("worker_id") or f"worker_{idx}_{model_name.lower()}"
            worker_id = re.sub(r"[^a-zA-Z0-9_\\-]", "_", worker_id)

            if worker_id in seen_worker_ids:
                worker_id = f"{worker_id}_{idx}"

            seen_worker_ids.add(worker_id)
            seen_models.add(model_name)

            cleaned_specs.append({
                "worker_id": worker_id,
                "model_name": model_name,
                "task_type": state["task_type"],
                "primary_metric": primary_metric,
                "rationale": spec.get("rationale", ""),
            })

        # Fallback: guarantee at least 3 workers.
        if len(cleaned_specs) < 3:
            fallback_models = ALLOWED_MODELS[state["task_type"]][:3]

            for model_name in fallback_models:
                if model_name in seen_models:
                    continue

                worker_id = f"worker_{model_name.lower()}"
                cleaned_specs.append({
                    "worker_id": worker_id,
                    "model_name": model_name,
                    "task_type": state["task_type"],
                    "primary_metric": primary_metric,
                    "rationale": "Fallback model added to guarantee at least three parallel workers.",
                })

                if len(cleaned_specs) >= 3:
                    break

        return {
            "primary_metric": primary_metric,
            "metric_reason": decision.get("metric_reason", ""),
            "worker_specs": cleaned_specs,
            "worker_results": [],
            "logs": [
                make_log(
                    "modeling_supervisor",
                    f"Selected metric={primary_metric}; created {len(cleaned_specs)} workers.",
                )
            ],
        }

    return modeling_supervisor_node


def assign_model_workers(state: ModelingState):
    """
    Dynamic parallel routing.

    Each Send creates a separate state for the same model_worker node.
    LangGraph supports this map-reduce style for dynamic parallel work. 
    """
    sends = []

    for worker_spec in state["worker_specs"]:
        sends.append(
            Send(
                "model_worker",
                {
                    **state,
                    "current_worker_spec": worker_spec,
                },
            )
        )

    return sends


def model_worker_node_factory(llm: Any):
    def model_worker_node(state: ModelingState) -> Dict[str, Any]:
        worker_spec = state["current_worker_spec"]
        worker_id = worker_spec["worker_id"]
        model_name = worker_spec["model_name"]
        print(f"\n[MODEL WORKER STARTED] worker_id={worker_id}, model={model_name}", flush=True)

        worker_dir = get_worker_artifact_dir(state, worker_id)
        code_path = worker_dir / "train_model.py"
        metrics_path = worker_dir / "metrics.json"
        model_path = worker_dir / "model.joblib"
        predictions_path = worker_dir / "predictions.csv"

        try:
            prompt = build_worker_code_prompt(state, worker_spec, worker_dir)
            raw_code = call_llm(llm, prompt)
            code = extract_python_code(raw_code)

            validate_generated_code_safety(code, str(worker_dir))

            with open(code_path, "w", encoding="utf-8") as f:
                f.write(code)

            execution = run_python_file(code_path)

            # One repair attempt.
            if execution["status"] != "success":
                repair_prompt = build_fix_code_prompt(
                    state=state,
                    worker_spec=worker_spec,
                    old_code=code,
                    stderr=execution["stderr"],
                    worker_dir=worker_dir,
                )

                raw_fixed_code = call_llm(llm, repair_prompt)
                fixed_code = extract_python_code(raw_fixed_code)

                validate_generated_code_safety(fixed_code, str(worker_dir))

                code = fixed_code

                with open(code_path, "w", encoding="utf-8") as f:
                    f.write(fixed_code)

                execution = run_python_file(code_path)

            if execution["status"] != "success":
                result: ModelWorkerResult = {
                    "worker_id": worker_id,
                    "model_name": model_name,
                    "status": "error",
                    "metrics": {},
                    "model_path": None,
                    "predictions_path": None,
                    "metrics_path": None,
                    "code_path": str(code_path),
                    "error": execution["stderr"][-4000:],
                    "generated_code": code,
                }

                return {
                    "worker_results": [result],
                    "logs": [
                        make_log(
                            worker_id,
                            f"Worker failed for model {model_name}.",
                            level="ERROR",
                        )
                    ],
                }

            if not metrics_path.exists():
                raise FileNotFoundError(
                    f"Worker finished but metrics.json was not created: {metrics_path}"
                )

            metrics = safe_json_load(metrics_path)

            result = {
                "worker_id": worker_id,
                "model_name": model_name,
                "status": "success",
                "metrics": metrics,
                "model_path": str(model_path) if model_path.exists() else None,
                "predictions_path": str(predictions_path) if predictions_path.exists() else None,
                "metrics_path": str(metrics_path),
                "code_path": str(code_path),
                "error": None,
                "generated_code": code,
            }

            return {
                "worker_results": [result],
                "logs": [
                    make_log(
                        worker_id,
                        f"Worker successfully trained {model_name}.",
                    )
                ],
            }

        except Exception as exc:
            result = {
                "worker_id": worker_id,
                "model_name": model_name,
                "status": "error",
                "metrics": {},
                "model_path": None,
                "predictions_path": None,
                "metrics_path": None,
                "code_path": str(code_path),
                "error": str(exc),
                "generated_code": None,
            }

            return {
                "worker_results": [result],
                "logs": [
                    make_log(
                        worker_id,
                        f"Worker crashed for model {model_name}: {str(exc)}",
                        level="ERROR",
                    )
                ],
            }

    return model_worker_node


def get_score_from_metrics(
    result: ModelWorkerResult,
    primary_metric: str,
    task_type: str,
) -> float:
    metrics = result.get("metrics", {})

    # Worker metrics schema may include validation_metrics and test_metrics.
    validation_metrics = metrics.get("validation_metrics", {})
    test_metrics = metrics.get("test_metrics", {})

    metric = normalize_metric_name(primary_metric)

    # Prefer validation metric for model selection to avoid test leakage.
    value = validation_metrics.get(metric)

    if value is None:
        value = test_metrics.get(metric)

    if value is None:
        # Some workers may use names like f1 instead of f1_weighted.
        aliases = {
            "f1_weighted": ["f1", "weighted_f1"],
            "precision_weighted": ["precision"],
            "recall_weighted": ["recall"],
            "rmse": ["root_mean_squared_error"],
            "mae": ["mean_absolute_error"],
            "r2": ["r2_score"],
        }

        for alias in aliases.get(metric, []):
            value = validation_metrics.get(alias)
            if value is not None:
                break
            value = test_metrics.get(alias)
            if value is not None:
                break

    if value is None:
        if task_type == "regression" and metric in {"mae", "rmse"}:
            return float("inf")
        return float("-inf")

    return float(value)


def judge_best_model_node(state: ModelingState) -> Dict[str, Any]:
    successful = [
        result for result in state.get("worker_results", [])
        if result.get("status") == "success"
    ]

    if not successful:
        return {
            "status": "failed",
            "reason": "No model worker finished successfully.",
            "logs": [
                make_log(
                    "judge",
                    "No successful model worker results.",
                    level="ERROR",
                )
            ],
        }

    primary_metric = state["primary_metric"]
    task_type = state["task_type"]

    lower_is_better = task_type == "regression" and primary_metric in {"mae", "rmse"}

    if lower_is_better:
        best = min(
            successful,
            key=lambda r: get_score_from_metrics(r, primary_metric, task_type),
        )
    else:
        best = max(
            successful,
            key=lambda r: get_score_from_metrics(r, primary_metric, task_type),
        )

    best_metrics = best.get("metrics", {})
    best_test_metrics = best_metrics.get("test_metrics", best_metrics)

    return {
        "best_model_name": best["model_name"],
        "best_worker_id": best["worker_id"],
        "best_model_path": best.get("model_path") or "",
        "best_predictions_path": best.get("predictions_path") or "",
        "best_metrics_path": best.get("metrics_path") or "",
        "best_metrics": best_test_metrics,
        "status": "best_model_selected",
        "logs": [
            make_log(
                "judge",
                f"Best model selected: {best['model_name']} from {best['worker_id']}.",
            )
        ],
    }


def should_tune_or_package(state: ModelingState) -> str:
    """
    Conditional router after judge.

    Returns:
        "tune" or "package"
    """
    if state.get("status") == "failed":
        return "package"

    retry_count = state.get("retry_count", 0)
    max_retries = state.get("max_retries", 1)

    if retry_count >= max_retries:
        return "package"

    task_type = state["task_type"]
    metric = state["primary_metric"]
    best_metrics = state.get("best_metrics", {})

    if task_type == "classification":
        value = best_metrics.get(metric)

        if value is None and metric == "f1_weighted":
            value = best_metrics.get("f1")

        if value is not None and float(value) < 0.65:
            return "tune"

    if task_type == "regression":
        r2 = best_metrics.get("r2")
        rmse = best_metrics.get("rmse")

        # If R2 is poor, try tuning.
        if r2 is not None and float(r2) < 0.30:
            return "tune"

        # If only RMSE exists, we do not know a universal threshold,
        # so packaging is safer.
        if r2 is None and rmse is not None:
            return "package"

    return "package"


def tuning_worker_node_factory(llm: Any):
    def tuning_worker_node(state: ModelingState) -> Dict[str, Any]:
        tuning_dir = ensure_dir(Path(state["artifacts_dir"]) / state["run_id"] / "tuning")
        code_path = tuning_dir / "tune_model.py"
        metrics_path = tuning_dir / "metrics.json"
        model_path = tuning_dir / "model.joblib"
        predictions_path = tuning_dir / "predictions.csv"

        try:
            prompt = build_tuning_code_prompt(state, tuning_dir)
            raw_code = call_llm(llm, prompt)
            code = extract_python_code(raw_code)

            validate_generated_code_safety(code, str(tuning_dir))

            with open(code_path, "w", encoding="utf-8") as f:
                f.write(code)

            execution = run_python_file(code_path, timeout_seconds=240)

            if execution["status"] != "success":
                return {
                    "tuning_used": False,
                    "tuning_status": "failed",
                    "retry_count": state.get("retry_count", 0) + 1,
                    "logs": [
                        make_log(
                            "tuning_worker",
                            f"Tuning failed: {execution['stderr'][-1000:]}",
                            level="ERROR",
                        )
                    ],
                }

            if not metrics_path.exists():
                return {
                    "tuning_used": False,
                    "tuning_status": "failed_no_metrics",
                    "retry_count": state.get("retry_count", 0) + 1,
                    "logs": [
                        make_log(
                            "tuning_worker",
                            "Tuning finished but metrics.json was not created.",
                            level="ERROR",
                        )
                    ],
                }

            tuned_metrics_payload = safe_json_load(metrics_path)
            tuned_test_metrics = tuned_metrics_payload.get(
                "test_metrics",
                tuned_metrics_payload,
            )

            # Compare tuned model to current best on primary metric.
            primary_metric = state["primary_metric"]
            task_type = state["task_type"]
            old_metrics = state.get("best_metrics", {})

            old_value = old_metrics.get(primary_metric)
            new_value = tuned_test_metrics.get(primary_metric)

            if primary_metric == "f1_weighted":
                old_value = old_value if old_value is not None else old_metrics.get("f1")
                new_value = new_value if new_value is not None else tuned_test_metrics.get("f1")

            accept_tuned = False

            if old_value is None or new_value is None:
                # If comparison is unclear but tuning succeeded, accept tuned model
                # because it is still a valid artifact.
                accept_tuned = True
            elif task_type == "regression" and primary_metric in {"mae", "rmse"}:
                accept_tuned = float(new_value) <= float(old_value)
            else:
                accept_tuned = float(new_value) >= float(old_value)

            if accept_tuned:
                return {
                    "tuning_used": True,
                    "tuning_status": "accepted",
                    "retry_count": state.get("retry_count", 0) + 1,
                    "best_model_path": str(model_path),
                    "best_predictions_path": str(predictions_path),
                    "best_metrics_path": str(metrics_path),
                    "best_metrics": tuned_test_metrics,
                    "logs": [
                        make_log(
                            "tuning_worker",
                            "Tuned model accepted as new best model.",
                        )
                    ],
                }

            return {
                "tuning_used": True,
                "tuning_status": "not_improved",
                "retry_count": state.get("retry_count", 0) + 1,
                "logs": [
                    make_log(
                        "tuning_worker",
                        "Tuning completed but did not improve the selected metric.",
                    )
                ],
            }

        except Exception as exc:
            return {
                "tuning_used": False,
                "tuning_status": "error",
                "retry_count": state.get("retry_count", 0) + 1,
                "logs": [
                    make_log(
                        "tuning_worker",
                        f"Tuning crashed: {str(exc)}",
                        level="ERROR",
                    )
                ],
            }

    return tuning_worker_node


def packaging_node(state: ModelingState) -> Dict[str, Any]:
    run_dir = get_run_artifact_dir(state)
    final_dir = ensure_dir(run_dir / "final")

    summary_path = final_dir / "modeling_summary.json"
    final_metrics_path = final_dir / "metrics.json"
    final_predictions_path = final_dir / "predictions.csv"
    final_model_path = final_dir / "best_model.joblib"

    status = state.get("status", "unknown")

    if status == "failed":
        summary = {
            "agent": "modeling_agent",
            "status": "error",
            "summary": "No model was successfully trained.",
            "reason": state.get("reason"),
            "worker_results": state.get("worker_results", []),
            "logs": state.get("logs", []),
        }

        safe_json_dump(summary, summary_path)

        return {
            "modeling_summary_path": str(summary_path),
            "status": "error",
            "reason": state.get("reason"),
            "logs": [
                make_log(
                    "packaging",
                    "Packaged failed modeling result.",
                    level="ERROR",
                )
            ],
        }

    # Copy best artifacts into final folder.
    best_model_path = state.get("best_model_path")
    best_predictions_path = state.get("best_predictions_path")
    best_metrics_path = state.get("best_metrics_path")

    if best_model_path and Path(best_model_path).exists():
        shutil.copy2(best_model_path, final_model_path)

    if best_predictions_path and Path(best_predictions_path).exists():
        shutil.copy2(best_predictions_path, final_predictions_path)

    if best_metrics_path and Path(best_metrics_path).exists():
        shutil.copy2(best_metrics_path, final_metrics_path)
    else:
        safe_json_dump(state.get("best_metrics", {}), final_metrics_path)

    summary = {
        "agent": "modeling_agent",
        "status": "success",
        "skipped": False,
        "summary": (
            f"Modeling completed. Best model: {state.get('best_model_name')}. "
            f"Primary metric: {state.get('primary_metric')}."
        ),
        "decisions": {
            "primary_metric": state.get("primary_metric"),
            "metric_reason": state.get("metric_reason"),
            "worker_specs": state.get("worker_specs", []),
            "best_model_name": state.get("best_model_name"),
            "best_worker_id": state.get("best_worker_id"),
            "tuning_used": state.get("tuning_used", False),
            "tuning_status": state.get("tuning_status", "not_used"),
        },
        "worker_results": state.get("worker_results", []),
        "best_metrics": state.get("best_metrics", {}),
        "artifacts": {
            "best_model_path": str(final_model_path),
            "predictions_path": str(final_predictions_path),
            "metrics_path": str(final_metrics_path),
            "modeling_summary_path": str(summary_path),
        },
        "logs": state.get("logs", []),
        "reason": None,
    }

    safe_json_dump(summary, summary_path)

    return {
        "final_model_path": str(final_model_path),
        "predictions_path": str(final_predictions_path),
        "metrics_path": str(final_metrics_path),
        "modeling_summary_path": str(summary_path),
        "status": "success",
        "reason": None,
        "logs": [
            make_log(
                "packaging",
                "Final modeling artifacts packaged successfully.",
            )
        ],
    }


# ============================================================
# 6. Graph builder
# ============================================================

def build_modeling_graph(llm: Any):
    """
    Builds and compiles the LangGraph Modeling Agent.

    LangGraph notes:
    - StateGraph nodes read and write shared state.
    - Reducers are used for keys updated by parallel workers.
    - Send dynamically invokes the same worker node with different state slices.
    """

    graph = StateGraph(ModelingState)

    graph.add_node("modeling_supervisor", modeling_supervisor_node_factory(llm))
    graph.add_node("model_worker", model_worker_node_factory(llm))
    graph.add_node("judge_best_model", judge_best_model_node)
    graph.add_node("tuning_worker", tuning_worker_node_factory(llm))
    graph.add_node("packaging", packaging_node)

    graph.add_edge(START, "modeling_supervisor")

    # Supervisor dynamically creates parallel workers.
    graph.add_conditional_edges(
        "modeling_supervisor",
        assign_model_workers,
        ["model_worker"],
    )

    # After all parallel model_worker branches finish,
    # their worker_results are reduced and judge receives combined state.
    graph.add_edge("model_worker", "judge_best_model")

    graph.add_conditional_edges(
        "judge_best_model",
        should_tune_or_package,
        {
            "tune": "tuning_worker",
            "package": "packaging",
        },
    )

    graph.add_edge("tuning_worker", "packaging")
    graph.add_edge("packaging", END)

    return graph.compile()


# ============================================================
# 7. Public runner
# ============================================================

def run_modeling_agent(
    llm: Any,
    prepared_dataset_path: str,
    target_column: str,
    task_type: str,
    business_task: str = "",
    run_id: Optional[str] = None,
    artifacts_dir: str = "artifacts",
    data_summary: Optional[Dict[str, Any]] = None,
    prep_summary: Optional[Dict[str, Any]] = None,
    max_retries: int = 1,
) -> Dict[str, Any]:
    """
    Runs the full Modeling Agent and returns a unified agent response.

    Args:
        llm:
            LangChain-compatible chat model with .invoke(prompt).
        prepared_dataset_path:
            Path to prepared CSV file.
        target_column:
            Name of target column in CSV.
        task_type:
            "classification" or "regression".
        business_task:
            Short business description.
        run_id:
            Optional run ID. If None, generated automatically.
        artifacts_dir:
            Base directory for artifacts.
        data_summary:
            Optional upstream EDA summary.
        prep_summary:
            Optional upstream preprocessing summary.
        max_retries:
            Number of tuning attempts. Usually 1 is enough for project demo.

    Returns:
        Unified response dict compatible with your agent format.
    """

    if task_type not in {"classification", "regression"}:
        raise ValueError("task_type must be 'classification' or 'regression'.")

    if not Path(prepared_dataset_path).exists():
        raise FileNotFoundError(f"Prepared dataset not found: {prepared_dataset_path}")

    run_id = run_id or str(uuid.uuid4())

    initial_state: ModelingState = {
        "run_id": run_id,
        "prepared_dataset_path": prepared_dataset_path,
        "target_column": target_column,
        "task_type": task_type,
        "business_task": business_task,
        "artifacts_dir": artifacts_dir,
        "data_summary": data_summary or {},
        "prep_summary": prep_summary or {},
        "worker_results": [],
        "logs": [
            make_log(
                "modeling_agent",
                f"Modeling workflow started for run_id={run_id}.",
            )
        ],
        "tuning_used": False,
        "tuning_status": "not_used",
        "retry_count": 0,
        "max_retries": max_retries,
        "status": "running",
        "reason": None,
    }

    compiled_graph = build_modeling_graph(llm)
    final_state = compiled_graph.invoke(initial_state)

    summary_path = final_state.get("modeling_summary_path")

    if summary_path and Path(summary_path).exists():
        summary = safe_json_load(summary_path)

        return {
            "agent": "modeling_agent",
            "status": summary.get("status", final_state.get("status")),
            "skipped": False,
            "summary": summary.get("summary", "Modeling workflow completed."),
            "decisions": summary.get("decisions", {}),
            "artifacts": summary.get("artifacts", {}),
            "next_input": {
                "best_model_name": summary.get("decisions", {}).get("best_model_name"),
                "test_metrics": summary.get("best_metrics", {}),
                "metrics_path": summary.get("artifacts", {}).get("metrics_path"),
                "predictions_path": summary.get("artifacts", {}).get("predictions_path"),
                "best_model_path": summary.get("artifacts", {}).get("best_model_path"),
            },
            "reason": summary.get("reason"),
        }

    return {
        "agent": "modeling_agent",
        "status": final_state.get("status", "unknown"),
        "skipped": False,
        "summary": "Modeling workflow finished, but summary artifact was not found.",
        "decisions": {
            "primary_metric": final_state.get("primary_metric"),
            "best_model_name": final_state.get("best_model_name"),
        },
        "artifacts": {
            "best_model_path": final_state.get("final_model_path"),
            "metrics_path": final_state.get("metrics_path"),
            "predictions_path": final_state.get("predictions_path"),
            "modeling_summary_path": final_state.get("modeling_summary_path"),
        },
        "next_input": {
            "best_model_name": final_state.get("best_model_name"),
            "test_metrics": final_state.get("best_metrics", {}),
        },
        "reason": final_state.get("reason"),
    }

## Здесь я работаю с сэмплом из датасета, просто проверяю саму корректность работы агента, так что оставил только численные признаки и + наны заменил медианами и так далее. После того, как будут готовы другие части и агенты - можно будет использовать моделлинг уже для всех данных!!

In [30]:
import os
import pandas as pd
import numpy as np

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

FULL_DATASET_PATH = "//Users/gleb/Downloads/ai_agents_for_smadimo/rent-prediction-2025/airbnb_train.csv"
DATASET_PATH = "rent-prediction-2025/airbnb_train_sample_5k_numeric.csv"

TARGET_COLUMN = "price"
TASK_TYPE = "regression"

# 1. Load and sample
df = pd.read_csv(FULL_DATASET_PATH)
df = df.sample(n=5000, random_state=42).reset_index(drop=True)

print("Original sample shape:", df.shape)

# 2. Clean target
if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' not found. "
        f"Available columns: {df.columns.tolist()}"
    )

if df[TARGET_COLUMN].dtype == "object":
    df[TARGET_COLUMN] = (
        df[TARGET_COLUMN]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    df[TARGET_COLUMN] = pd.to_numeric(df[TARGET_COLUMN], errors="coerce")

df = df.dropna(subset=[TARGET_COLUMN])

# 3. Convert obvious date columns to numeric timestamps before numeric filtering
date_cols = ["host_since", "first_review", "last_review"]

for col in date_cols:
    if col in df.columns:
        converted = pd.to_datetime(df[col], errors="coerce")
        if converted.notna().sum() > 0:
            df[col] = converted.astype("int64") // 10**9
            df[col] = df[col].replace(-9223372037, np.nan)

# 4. Keep only numeric columns
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()

if TARGET_COLUMN not in numeric_cols:
    numeric_cols.append(TARGET_COLUMN)

df = df[numeric_cols]

# 5. Drop ID-like columns
id_like_cols = ["id", "host_id"]

df = df.drop(columns=[col for col in id_like_cols if col in df.columns])

# 6. Fill missing numeric values
df = df.replace([np.inf, -np.inf], np.nan)

feature_cols = [col for col in df.columns if col != TARGET_COLUMN]

for col in feature_cols:
    median_value = df[col].median()
    if pd.isna(median_value):
        median_value = 0
    df[col] = df[col].fillna(median_value)

df = df.dropna(subset=[TARGET_COLUMN])

# Final check
print("Prepared smoke-test shape:", df.shape)
print("Total missing values:", int(df.isna().sum().sum()))
print("Dtypes:")
print(df.dtypes)

df.to_csv(DATASET_PATH, index=False)
print("Saved to:", DATASET_PATH)

# 7. Create LLM

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key="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",
    base_url="https://openrouter.ai/api/v1",
    temperature=0.2,
)

# 8. Run Modeling Agent
result = run_modeling_agent(
    llm=llm,
    prepared_dataset_path=DATASET_PATH,
    target_column=TARGET_COLUMN,
    task_type=TASK_TYPE,
    business_task=(
        "Predict Airbnb rental price based on numeric listing characteristics. "
        "This is a standalone smoke test of the Modeling Agent."
    ),
    run_id="airbnb_modeling_test_5k_numeric",
    artifacts_dir="artifacts",
    data_summary={
        "n_rows": int(df.shape[0]),
        "n_columns": int(df.shape[1]),
        "columns": df.columns.tolist(),
        "target_column": TARGET_COLUMN,
        "missing_values": df.isna().sum().to_dict(),
        "dtypes": df.dtypes.astype(str).to_dict(),
        "note": "Smoke test dataset: numeric features only, no missing values."
    },
    prep_summary={
        "mode": "standalone_modeling_test_numeric_only",
        "note": (
            "For isolated testing of the Modeling Agent, the dataset was reduced "
            "to 5000 rows and numeric columns only."
        ),
        "dropped_columns": "All non-numeric columns plus ID-like columns.",
        "missing_strategy": "Numeric missing values filled with median, fallback to 0.",
        "datetime_strategy": "host_since, first_review and last_review converted to Unix timestamp.",
    },
    max_retries=1,
)

print("\n=== MODELING RESULT ===")
print(result)

print("\nArtifacts:")
for key, value in result.get("artifacts", {}).items():
    print(f"{key}: {value}")

Original sample shape: (5000, 52)
Prepared smoke-test shape: (5000, 31)
Total missing values: 0
Dtypes:
host_since                     float64
host_listings_count            float64
host_total_listings_count      float64
latitude                       float64
longitude                      float64
accommodates                     int64
bathrooms                      float64
bedrooms                       float64
beds                           float64
price                          float64
availability_30                  int64
availability_60                  int64
availability_90                  int64
availability_365                 int64
number_of_reviews                int64
number_of_reviews_ltm            int64
number_of_reviews_l30d           int64
availability_eoy               float64
number_of_reviews_ly           float64
estimated_occupancy_l365d      float64
estimated_revenue_l365d        float64
first_review                   float64
last_review                    float64

/var/folders/2k/dgh4_8m15p140bw346vmwk4r0000gn/T/ipykernel_68902/502448212.py:170: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat(timespec="seconds") + "Z"



[MODEL WORKER STARTED] worker_id=worker_1, model=LinearRegression
[MODEL WORKER STARTED] worker_id=worker_2, model=RandomForestRegressor


[MODEL WORKER STARTED] worker_id=worker_3, model=GradientBoostingRegressor


/var/folders/2k/dgh4_8m15p140bw346vmwk4r0000gn/T/ipykernel_68902/502448212.py:170: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat(timespec="seconds") + "Z"
/var/folders/2k/dgh4_8m15p140bw346vmwk4r0000gn/T/ipykernel_68902/502448212.py:170: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat(timespec="seconds") + "Z"



=== MODELING RESULT ===
{'agent': 'modeling_agent', 'status': 'success', 'skipped': False, 'summary': 'Modeling completed. Best model: RandomForestRegressor. Primary metric: rmse.', 'decisions': {'primary_metric': 'rmse', 'metric_reason': "RMSE is appropriate for regression tasks where large errors are significant, as it provides a clear measure of the model's performance in predicting rental prices.", 'worker_specs': [{'worker_id': 'worker_1', 'model_name': 'LinearRegression', 'task_type': 'regression', 'primary_metric': 'rmse', 'rationale': 'Linear Regression serves as a simple baseline model, providing a straightforward approach to understand the relationship between features and the target price.'}, {'worker_id': 'worker_2', 'model_name': 'RandomForestRegressor', 'task_type': 'regression', 'primary_metric': 'rmse', 'rationale': 'Random Forest is a tree-based model that can capture complex interactions between features and is robust to overfitting, making it suitable for this datas

In [31]:
import os
import json
import shutil
import datetime
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate


# ======================
# 🔧 UTILS
# ======================

def load_json(path):
    with open(path) as f:
        return json.load(f)


def save_text(path, text):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(text)


def copy_file(src, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(src, dst)


def exists(path):
    return os.path.exists(path)


def safe_parse_json(text):
    """Парсим JSON из ответа LLM"""
    try:
        # Ищем JSON блок в ответе
        start = text.find('{')
        end = text.rfind('}') + 1
        if start >= 0 and end > start:
            json_str = text[start:end]
            return json.loads(json_str)
        raise ValueError("JSON не найден в ответе")
    except Exception as e:
        raise ValueError(f"Ошибка парсинга JSON: {e}\nRAW:\n{text}")


# ======================
# 📂 LOAD CURRENT MODELS
# ======================

def load_current_models(workers_dir):
    models = []

    for worker_name in os.listdir(workers_dir):
        worker_path = os.path.join(workers_dir, worker_name)

        if not os.path.isdir(worker_path):
            continue

        metrics_path = os.path.join(worker_path, "metrics.json")
        model_path = os.path.join(worker_path, "model.joblib")

        if not os.path.exists(metrics_path):
            print(f"⚠️ Нет metrics.json в {worker_name}")
            continue

        metrics = load_json(metrics_path)

        models.append({
            "worker_id": worker_name,
            "full_metrics": metrics,  # Весь JSON как есть
            "model_path": model_path,
            "metrics_path": metrics_path
        })

    return models


# ======================
# 🧠 LLM
# ======================

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key="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",
    base_url="https://openrouter.ai/api/v1",
    temperature=0.2,
)

# Prompt для анализа моделей и генерации отчёта
prompt = ChatPromptTemplate.from_messages([
    ("system", """
Ты ML Model Registry Agent. Твоя задача:

1. Проанализировать метрики текущих моделей и глобальную модель
2. Выбрать лучшую модель (по RMSE на тестовой выборке - меньше лучше)
3. Сгенерировать подробный markdown отчёт

Вернуть JSON с двумя частями:

{{
  "decision": {{
    "best_worker_id": "worker_X",
    "best_model_name": "RandomForestRegressor",
    "action": "update_global_model" или "keep_old_model",
    "improvement_percent": 12.5
  }},
  "report_markdown": "# 📊 Отчёт о сравнении ML моделей\\n\\n..."
}}

ПРАВИЛА ОТЧЁТА:
- Заголовок с датой
- Таблица со всеми текущими моделями и их метриками
- Лучшая модель выделена звёздочкой ⭐
- Раздел "Результаты сравнения" с деталями улучшения
- Подробное описание каждой модели
- Рекомендации по действиям

ПРАВИЛА РЕШЕНИЯ:
- Сравни по RMSE (меньше = лучше)
- Если глобальной модели нет → update_global_model
- Если новая тестовая RMSE < старой тестовой RMSE → update_global_model
- Иначе → keep_old_model
- improvement_percent = ((old_rmse - new_rmse) / old_rmse) * 100

Генерируй КРАСИВЫЙ и ИНФОРМАТИВНЫЙ markdown отчёт!
"""),

    ("user", """
CURRENT MODELS:
{current_models}

GLOBAL MODEL:
{global_model}
""")
])


# ======================
# 🤖 AGENT
# ======================

def model_registry_agent(state):
    try:
        workers_dir = state["workers_dir"]

        if not os.path.exists(workers_dir):
            raise ValueError(f"workers_dir не найден: {workers_dir}")

        print("📂 Loading models...")

        current_models = load_current_models(workers_dir)

        if len(current_models) == 0:
            raise ValueError("Не найдено ни одной модели")

        print("✅ Models loaded:", len(current_models))

        # --- глобальная модель ---
        global_metrics_path = "artifacts/global_best/metrics.json"
        global_model_path = "artifacts/global_best/model.joblib"

        if exists(global_metrics_path):
            global_model = load_json(global_metrics_path)
            print(f"✅ Глобальная модель найдена: {global_model.get('model_name', 'Unknown')}")
        else:
            global_model = None
            print(f"⚠️ Глобальной модели не найдено (первый запуск)")

        print("📊 Calling LLM...")

        chain = prompt | llm

        response = chain.invoke({
            "current_models": json.dumps(current_models, indent=2, ensure_ascii=False),
            "global_model": json.dumps(global_model, indent=2, ensure_ascii=False) if global_model else "None"
        })

        print("🧠 RAW LLM RESPONSE:\n", response.content)

        result = safe_parse_json(response.content)
        
        decision = result.get("decision", {})
        report_markdown = result.get("report_markdown", "")

        # --- найти лучший worker ---
        best_worker_id = decision.get("best_worker_id")
        best_worker = next(
            (w for w in current_models if w["worker_id"] == best_worker_id),
            None
        )
        
        if not best_worker:
            raise ValueError(f"Worker {best_worker_id} не найден")

        # --- обновление глобальной ---
        os.makedirs("artifacts/global_best", exist_ok=True)

        global_updated = False
        action = decision.get("action", "keep_old_model")

        if action == "update_global_model":
            # Копируем саму модель (joblib файл)
            if best_worker["model_path"] and exists(best_worker["model_path"]):
                copy_file(best_worker["model_path"], global_model_path)
                print(f"📦 Модель скопирована: {best_worker['model_path']} → {global_model_path}")
            else:
                print(f"⚠️ Model файл не найден: {best_worker['model_path']}")
            
            # Копируем метрики (JSON)
            copy_file(best_worker["metrics_path"], global_metrics_path)
            print(f"📊 Метрики скопированы: {best_worker['metrics_path']} → {global_metrics_path}")
            
            global_updated = True
            print(f"✅ Глобальная модель обновлена: {decision.get('best_model_name')} ({best_worker_id})")
        else:
            print(f"⛔ Глобальная модель не обновлена (осталась текущая: {global_model.get('model_name', 'Unknown') if global_model else 'нет'})")

        # --- сохраняем отчёт ---
        report_path = f"artifacts/final/model_report_{state['run_id']}.md"
        save_text(report_path, report_markdown)
        print(f"📄 Отчёт сохранён: {report_path}")

        # --- state ---
        state.setdefault("artifacts", {})
        state.setdefault("modeling_summary", {})
        state.setdefault("logs", [])

        state["artifacts"]["final_report_path"] = report_path

        state["modeling_summary"]["global_comparison"] = decision

        state["logs"].append({
            "step": "model_registry_agent",
            "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "status": "success"
        })

        return {
            "agent": "model_registry_agent",
            "status": "success",
            "skipped": False,
            "summary": f"Лучшая модель: {decision.get('best_model_name')}. Действие: {action}. Улучшение: {decision.get('improvement_percent', 0):.2f}%",
            "decisions": decision,
            "artifacts": {
                "final_report_path": report_path,
                "global_model_path": global_model_path,
                "global_updated": global_updated
            },
            "next_input": {},
            "reason": None
        }

    except Exception as e:
        print("❌ ERROR:", str(e))
        import traceback
        traceback.print_exc()

        return {
            "agent": "model_registry_agent",
            "status": "error",
            "skipped": False,
            "summary": "Ошибка при сравнении моделей",
            "decisions": {},
            "artifacts": {},
            "next_input": {},
            "reason": str(e)
        }


# ======================
# ▶️ LOCAL RUN
# ======================

if __name__ == "__main__":
    state = {
        "run_id": "test_run",
        "workers_dir": "/Users/gleb/Downloads/ai_agents_for_smadimo/artifacts/airbnb_modeling_test_5k_numeric/workers"
    }

    result = model_registry_agent(state)

    print("\n🎯 FINAL RESULT:\n")
    print(json.dumps(result, indent=2, ensure_ascii=False))

📂 Loading models...
✅ Models loaded: 3
✅ Глобальная модель найдена: GradientBoostingRegressor
📊 Calling LLM...
🧠 RAW LLM RESPONSE:
 ```json
{
  "decision": {
    "best_worker_id": "worker_2",
    "best_model_name": "RandomForestRegressor",
    "action": "update_global_model",
    "improvement_percent": 0.0065
  },
  "report_markdown": "# 📊 Отчёт о сравнении ML моделей\n\n## Дата: 2023-10-01\n\n### Таблица с метриками моделей\n| Worker ID | Модель                     | RMSE (тест) | MAE (тест) | R² (тест) |\n|-----------|----------------------------|-------------|------------|-----------|\n| worker_1 | LinearRegression           | 173.41      | 112.87     | 0.279     |\n| worker_2 | RandomForestRegressor ⭐    | 125.68      | 82.90      | 0.621     |\n| worker_3 | GradientBoostingRegressor  | 125.69      | 84.73      | 0.621     |\n\n### Результаты сравнения\nНа основе анализа метрик моделей, лучшей моделью является **RandomForestRegressor** от `worker_2`, с RMSE 125.68, что на 0.0065% л